# Offline Analysis Demo (Second Pass)

This notebook demonstrates a reusable offline workflow migrated from your draft `Analysis.py`/`Fitting.py` ideas into production code.

Core module used: `rheed_core.core.offline_analysis`

## Workflow
1. Prepare time/intensity arrays
2. Denoise signal (`median` + `FFT` + smooth)
3. Detect cycle boundaries using camera/laser prior
4. Split and process each cycle (tail tune + linear background removal)
5. Estimate cycle-wise relaxation time `tau`


In [1]:
import numpy as np

from rheed_core.core.offline_analysis import (
    analyze_rheed_signal,
    detect_cycle_boundaries,
    preprocess_signal,
    select_range,
)


In [2]:
# Step 1: Synthetic data placeholder. Replace with your exported TSST arrays later.
camera_freq = 20.0  # Hz
laser_freq = 0.5    # Hz (one pulse every 2s)
period = 1.0 / laser_freq
tau_true = 0.45

dt = 1.0 / camera_freq
t = np.arange(0.0, 60.0, dt)
phase = np.mod(t, period)
rng = np.random.default_rng(10)
y = 1.0 - np.exp(-phase / tau_true) + 0.03 * rng.standard_normal(t.shape[0])

# Optional: analyze only a target window.
window = select_range(np.stack([t, y], axis=1), start=5.0, end=55.0, y_col=1)
t_win, y_win = window[:, 0], window[:, 1]
print("Window length:", len(t_win))


Window length: 999


In [3]:
# Step 2: Preprocess
t_proc, y_proc = preprocess_signal(
    t_win,
    y_win,
    sample_rate_hz=camera_freq,
    median_kernel_size=5,
    fft_band=(0.05, 4.0),
    smooth_window=5,
)

# Step 3: Cycle boundaries
peaks = detect_cycle_boundaries(
    y_proc,
    camera_freq=camera_freq,
    laser_freq=laser_freq,
    convolve_step=5,
    prominence=0.03,
)
print("Detected boundaries:", len(peaks))


Detected boundaries: 25


In [4]:
# Step 4 + 5: Full per-cycle analysis with tau fit
fits = analyze_rheed_signal(
    t_proc,
    y_proc,
    camera_freq=camera_freq,
    laser_freq=laser_freq,
    convolve_step=5,
    prominence=0.03,
    tune_tail=True,
    trim_first=0,
    linear_ratio=0.8,
    fit_mode="growth",
)

taus = [f.tau.tau_s for f in fits if f.tau.tau_s is not None]
print("Total cycles:", len(fits))
print("Tau count:", len(taus))
print("Tau median:", None if not taus else float(np.median(taus)))
print("Tau truth:", tau_true)


Total cycles: 24
Tau count: 24
Tau median: 0.40630486367938606
Tau truth: 0.45


## Plug in your real data
- Replace synthetic `t_win`, `y_win` with arrays parsed from your draft exports (`txt`, `h5`, etc.).
- Keep `camera_freq` and `laser_freq` consistent with experiment settings.
- Tune `prominence` and `convolve_step` until cycle boundaries align with known pulse periods.
- Compare `tau` trend to known growth regime transitions.
